# Gas Giant Generator

This notebook uses the `SphereCamera` class from `render/camera.py` to render gas giants with rings.

In [ ]:
# Environment Setup
import os
import sys

# Add project root to path so we can import modules correctly
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print(f"Project root added to sys.path: {ROOT_DIR}")

# Install dependencies if not present (helpful for Colab)
try:
    import taichi as ti
    import numpy as np
    from PIL import Image as PILImage
    print("Dependencies already installed.")
except ImportError:
    print("Installing dependencies...")
    !pip install -q taichi numpy Pillow
    import taichi as ti
    import numpy as np
    from PIL import Image as PILImage

print(f"Taichi version: {ti.__version__}")

In [ ]:
import taichi as ti
import numpy as np
from render.camera import SphereCamera
from IPython.display import display, Image

# Initialize Taichi
ti.init(arch=ti.gpu if ti.lang.cpu.get_custom_config().arch == ti.gpu else ti.cpu)

RES = 512
camera = SphereCamera(fluid_res=RES, render_res=RES, has_rings=1, samples=4)

# Create a dummy fluid buffer (or use actual fluid sim results)
render_buffer = ti.Vector.field(3, dtype=float, shape=(RES, RES))

@ti.kernel
def init_buffer(buf: ti.template()):
    for i, j in buf:
        # Generate simple ammonia-band-like texture
        buf[i, j] = ti.Vector([0.8, 0.6, 0.4]) * (ti.sin(float(j) * 0.1) * 0.2 + 0.8)

init_buffer(render_buffer)

print("Rendering gas giant...")
camera.render_gas_giant(render_buffer, 1.0, 1.0, -1.0)

img = camera.get_image_data()
import PIL.Image
display(PIL.Image.fromarray((img * 255).astype(np.uint8)))
print("Done!")